# 혜진 v6: 팀원 실험을 참고한 암종별 선별·넓은 트리 입력 비교
이 노트북 하나에서 원본 데이터 읽기, 변이 파싱, 학습 안에서 변수 선별, 동일 그룹 CV, 선택 고정 후 재분할 확인, 학습·피클·CSV·보고서 저장까지 실행합니다. 이전 피클이나 프로젝트 Python 파일은 필요하지 않습니다. 공유 data/와 requirements-v6.txt의 환경이 필요합니다.

팀원 문서는 실험 아이디어입니다. 다른 분할의 0.4965를 우리 검증 점수와 직접 비교하지 않습니다. 테스트 예측을 실제 정답이라고 확정하거나, 암종 비율을 억지로 맞추거나, test 빈도로 변이를 제거하지 않습니다. v5에 이미 적용한 직접 트리 입력·변이 부담 정보 보존을 유지하고 선별 방식을 비교합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v6';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v6 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


## 데이터 EDA
클래스 크기, 환자별 변이 유전자 수, 결측을 확인합니다. test 요약은 분포를 설명하는 용도이며 사전·선별·학습에 적합하지 않습니다.

In [2]:
# Descriptive EDA only: no test-derived vocabularies or fitted preprocessing.
burden_train=(train[genes].notna()&train[genes].ne('WT')).sum(axis=1)
burden_test=(test[genes].notna()&test[genes].ne('WT')).sum(axis=1)
eda=pd.DataFrame({'train':burden_train.describe(),'test':burden_test.describe()})
eda.to_csv(OUT/'eda_mutated_gene_burden.csv')
byclass=pd.DataFrame({'SUBCLASS':y,'mutated_genes':burden_train}).groupby('SUBCLASS').mutated_genes.agg(['count','mean','median','max'])
byclass.to_csv(OUT/'eda_by_class.csv')
missing=pd.DataFrame({'train_missing':train[genes].isna().sum(),'test_missing':test[genes].isna().sum()})
missing.to_csv(OUT/'eda_missing_by_gene.csv')
print('4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.')
display(eda);display(byclass)


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v6 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241
4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.
             train         test
count  6201.000000  2546.000000
mean     35.299629    78.134328
std     100.776379   213.964535
min       0.000000     0.000000
25%       7.000000    11.000000
50%      14.000000    28.000000
75%      27.000000    52.000000
max    2393.000000  2784.000000
          count        mean  median   max
SUBCLASS                                 
ACC          72   39.430556    24.0   396
BLCA        104   70.490385    54.0   382
BRCA        786   17.437659     9.0   853
CESC        155   44.129032    20.0   480
COAD        223   73.121076    23.0  1631
DLBC         38   67.421053    30.0   704
GBMLGG      461   12.535792    11.0   121
HNSC        223   40.569507    31.0   386
KIPAN       515   16.0

## 1. 기존 파싱과 v5 고정 비교군
동의 변이를 무조건 무해하다고 단정하지 않습니다. 학습 지원도는 암 관련성의 의학적 판정이 아닙니다. 원본 프로필 그룹을 분리하며 사전·선별·스케일러는 각 학습 fold만 봅니다.

In [3]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [4]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [5]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


## 2. 새 선별법과 특이도 가중치
- global: ANOVA 전체 순위로 4,096개. v5보다 입력을 넓혀 약한 협동 신호가 잘렸는지 점검합니다.
- balanced: 26개 암종 각각 one-vs-rest chi2를 구하고 **해당 암에서 더 자주 관측되는 변수** 순위를 번갈아 뽑습니다. 전체 순위가 소수 암종의 표지를 밀어내는지 점검합니다. 트리 4,096개, 선형 2,048개입니다.
- specificity: 학습 행에서 `P(암종 | 변수)`가 전체 암종 비율에서 얼마나 벗어나는지 KL divergence로 구하고 관측 수/(관측 수+20)으로 작은 표본을 줄입니다. 생물학적 드라이버라는 뜻은 아닙니다. 그 가중치로 변이 블록을 L2 정규화하되 변이 개수·유형 비율·경로 요약은 별도로 보존합니다.

신규 선택에서는 학습 값이 완전히 같은 열만 SHA256으로 중복 제거합니다. v5 대조군의 Pearson 선별은 그대로 유지됩니다. 숫자·경로 요약은 신규 선별에서 항상 보존합니다. 초기 8개 설정과 사전에 정한 v5 혼합 비율만 비교합니다.

초기 8개 설정을 본 뒤, v5에서 쓰던 희귀 변이 증강과 넓은 트리 입력을 결합한 4개 설정을 추가했습니다. 총 12개 설정의 적응 탐색이므로 개발 데이터 선택 편향이 있습니다. Run All은 12개를 처음부터 계산합니다. 증강은 학습 관측 변이만 사용하며 원본/증강 가중치 1/0.5, 평균 10개 추가입니다. 선별·사전·스케일러는 원본으로만 적합합니다.

In [6]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8


Total new configurations including augmentation extension: 12


## 3. 同一 그룹 3-fold 비교
선택 점수는 v5와 같은 clean 0.6 + 희귀 관측 변이 추가 0.3 + 중복 토큰 0.1입니다. 합성 스트레스는 실제 test 분포나 질병 불변성을 보장하지 않습니다. 원본 검증 macro F1도 함께 제한합니다.

In [7]:
rows6=[];P6={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v5_reference',*NEW]}
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,SEED+901+fold)
 ms,ws=fit_base(tr)
 for j,f in enumerate(frames):P6['v5_reference'][j,b]=predict_members(ms,ws,f)
 del ms
 cache={}
 for name in NEW:
  t=time.time();m=fit_new6(name,tr,cache)
  for j,ev in enumerate(evs):P6[name][j,b]=m.predict_rows(m.encoder.transform(ev))
  print('v6',fold,name,'seconds',round(time.time()-t,1),flush=True)
  del m
 for name,p in P6.items():
  fs=[f1_score(yi[va],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  rows6.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(rows6).to_csv(OUT/'cv_folds.csv',index=False)
 np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**P6)
 print('Completed fold',fold,flush=True)

def evaluate6(p,splits):
 fs=np.array([[f1_score(yi[dev[b]],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)] for _,b in splits])
 return dict(clean=fs[:,0].mean(),noise=fs[:,1].mean(),duplicate=fs[:,2].mean(),score=(fs@np.array([.6,.3,.1])).mean(),clean_std=fs[:,0].std(ddof=1))
PLANS6={'v5_reference':[('v5_reference',1.)]}
for name in NEW:
 PLANS6[name]=[(name,1.)]
 for w in [.25,.5]:PLANS6[f'v5plus_{name}_{w}']=[('v5_reference',1-w),(name,w)]
rank6=pd.DataFrame({n:evaluate6(sum(P6[k]*w for k,w in plan),folds) for n,plan in PLANS6.items()}).T.sort_values('score',ascending=False)
base=rank6.loc['v5_reference'];eligible=rank6[(rank6.clean>=base.clean-.002)&(rank6.score>=base.score+.002)]
winner6=eligible.index[0] if len(eligible) else 'v5_reference';plan6=PLANS6[winner6]
rank6.to_csv(OUT/'model_comparison.csv');(OUT/'selection.json').write_text(json.dumps(dict(winner=winner6,plan=plan6,passes_primary=winner6!='v5_reference'),indent=2))
display(rank6.head(15));print('Frozen candidate:',winner6,flush=True)


Cached completed 8-setting CV + four additional actual fits per fold.
Frozen candidate: v5plus_gated_s3_aug_global_lgb4096_0.25


,clean,noise,duplicate,score,clean_std
v5plus_gated_s3_aug_global_lgb4096_0.25,0.435576,0.345775,0.433301,0.408408,0.009199
v5plus_balanced_lr_C0.5_0.25,0.438477,0.331850,0.435983,0.406240,0.013024
v5plus_gated_s3_aug_balanced_lgb4096_0.25,0.434247,0.341130,0.432971,0.406184,0.011446
v5plus_balanced_lr_C0.1_0.25,0.435671,0.335636,0.434887,0.405582,0.006421
v5_reference,0.432215,0.339619,0.433093,0.404524,0.002071
v5plus_gated_s3_aug_balanced_lgb4096_0.5,0.429433,0.341126,0.430722,0.403070,0.013346
v5plus_gated_s3_aug_global_lgb4096_0.5,0.426017,0.349388,0.425997,0.403026,0.009342
v5plus_specificity_lr_C1.0_0.25,0.430861,0.331596,0.428449,0.400840,0.011204
v5plus_raw_aug_global_xgb4096_d3_0.25,0.431276,0.330471,0.427691,0.400676,0.003831
gated_s3_aug_global_lgb4096,0.426273,0.339723,0.426273,0.400308,0.006798


## 4. 선택 고정 후 다른 그룹 분할 확인
같은 개발 환자를 seed=57로 다시 나누므로 독립 새 환자 검증은 아닙니다. 여기서 후보나 비율을 다시 튜닝하지 않습니다. 점수가 v5 이상이며 clean 하락이 0.002 이하여야 채택합니다.

In [8]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)
confirm6=[]
newfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=57).split(X.iloc[dev],y[dev],groups[dev]))
for fold,(a,b) in enumerate(newfolds):
 tr=dev[a];va=dev[b];frames,_,_=eval_views(va,tr,SEED+1901+fold)
 old=fit_base(tr);frozen=fit_plan6(tr,plan6,old)
 for name,(ms,ws) in [('v5_reference',old),('v6_frozen',frozen)]:
  fs=[f1_score(yi[va],predict_members(ms,ws,f).argmax(axis=1),average='macro',zero_division=0) for f in frames]
  confirm6.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(confirm6).to_csv(OUT/'confirmation_cv.csv',index=False)
 print('Confirmation fold',fold,confirm6[-2:],flush=True)
 del old,frozen
conf6=pd.DataFrame(confirm6).groupby('model')[['clean','noise','duplicate','score']].mean()
accepted6=bool(winner6!='v5_reference' and conf6.loc['v6_frozen','score']>=conf6.loc['v5_reference','score'] and conf6.loc['v6_frozen','clean']>=conf6.loc['v5_reference','clean']-.002)
final_plan6=plan6 if accepted6 else PLANS6['v5_reference']
print('Accepted new candidate:',accepted6);display(conf6)


Confirmation fold 0 [{'model': 'v5_reference', 'fold': 0, 'clean': 0.4271011635887081, 'noise': 0.324677525880574, 'duplicate': 0.4333004795819449, 'score': 0.39699400387559153}, {'model': 'v6_frozen', 'fold': 0, 'clean': 0.42920253393686286, 'noise': 0.32300293470186403, 'duplicate': 0.42893527496197514, 'score': 0.39731592826887446}]
Confirmation fold 1 [{'model': 'v5_reference', 'fold': 1, 'clean': 0.4320547919591109, 'noise': 0.33813979463312793, 'duplicate': 0.43743215045768297, 'score': 0.40441802861117315}, {'model': 'v6_frozen', 'fold': 1, 'clean': 0.43724565051859093, 'noise': 0.34819024841329704, 'duplicate': 0.4413370728330271, 'score': 0.4109381721184464}]
Confirmation fold 2 [{'model': 'v5_reference', 'fold': 2, 'clean': 0.4255393293508733, 'noise': 0.31681986833514786, 'duplicate': 0.42301892444374994, 'score': 0.39267145055544334}, {'model': 'v6_frozen', 'fold': 2, 'clean': 0.4252715626731903, 'noise': 0.3279484119578603, 'duplicate': 0.42770568150720467, 'score': 0.3963

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac

,clean,noise,duplicate,score
model,,,,
v5_reference,0.428232,0.326546,0.431251,0.398028
v6_frozen,0.430573,0.333047,0.432659,0.401524


## 5. 홀드아웃 재평가·전체 학습·모델 저장
이 홀드아웃은 이전 버전에서 관측한 데이터이므로 최종 독립 성능이 아닙니다. 채택 실패 시 v5를 유지하며 실패도 기록합니다. 테스트 정답은 학습·선택에 쓰지 않습니다.

In [9]:
hm,hw=fit_plan6(dev,final_plan6);hp=predict_members(hm,hw,X.iloc[hold]);pred=labels[hp.argmax(axis=1)]
metrics=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2))
hreport=pd.DataFrame(classification_report(y[hold],pred,output_dict=True,zero_division=0)).T;hreport.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y[hold],'prediction':pred}).to_csv(OUT/'holdout_predictions.csv',index=False)
print('Previously observed holdout:',metrics);del hm
models,weights=fit_plan6(np.arange(len(train)),final_plan6)
feature_rows=[];sizes=[]
for m,w in zip(models,weights):
 for f in m.raw_names:feature_rows.append({'model':m.name,'view':m.view,'feature':f})
 params=int(sum(z.size for z in m.est.coefs_)+sum(z.size for z in m.est.intercepts_)) if hasattr(m.est,'coefs_') else (int(m.est.coef_.size+m.est.intercept_.size) if hasattr(m.est,'coef_') else None)
 sizes.append({'model':m.name,'view':m.view,'weight':float(w),'inputs':len(m.names),'classifier_parameters':params,'unknown_weight':m.encoder.unknown if m.encoder else None,'silent_weight':m.encoder.silent if m.encoder else None,'support':m.encoder.support if m.encoder else None,'training_iterations':np.asarray(m.est.n_iter_).tolist() if hasattr(m.est,'n_iter_') else None})
features=pd.DataFrame(feature_rows);features.to_csv(OUT/'final_features.csv',index=False)
(OUT/'model_sizes.json').write_text(json.dumps(sizes,indent=2))
def predict_bundle(bundle,frame):return predict_members(bundle['models'],bundle['weights'],frame)
bundle={'version':6,'models':models,'weights':weights,'genes':genes,'classes':labels,'winner':winner6,'plan':final_plan6,'holdout_metrics':metrics,'sizes':sizes,'predict':predict_bundle}
model_path=OUT/'cancer_model_v6.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=loaded['predict'](loaded,test)
assert tp.shape==(len(test),NC) and np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
submission=sample.copy();submission.SUBCLASS=labels[tp.argmax(axis=1)]
assert submission.ID.equals(sample.ID) and submission.SUBCLASS.isin(labels).all()
submission.to_csv(OUT/'submission_v6.csv',index=False);submission.to_csv(SUBMIT/'혜진_submission_v6.csv',index=False)
pd.DataFrame(tp,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL:',model_path.resolve(),'MiB:',round(model_path.stat().st_size/1024**2,2));display(FileLink(str(model_path)))
print('SUBMISSION:',(SUBMIT/'혜진_submission_v6.csv').resolve());display(pd.DataFrame(sizes))


Previously observed holdout: {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}
MODEL: /Users/mac/project/cancer_project/artifacts/v6/cancer_model_v6.pkl MiB: 83.07
SUBMISSION: /Users/mac/project/cancer_project/submission/혜진_submission_v6.csv


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac

/Users/mac/project/cancer_project/artifacts/v6/cancer_model_v6.pkl

,model,view,weight,inputs,classifier_parameters,unknown_weight,silent_weight,support,training_iterations
0,legacy_lr,legacy,0.062500,2048,53274.0,NaN,NaN,NaN,[34]
1,legacy_lgb,legacy,0.062500,2048,NaN,NaN,NaN,NaN,250
2,legacy_extra,legacy,0.062500,1024,NaN,NaN,NaN,NaN,None
3,gated_s3_aug_lgb,gated_syn_down_s3,0.140625,2048,NaN,0.25,0.25,3.0,300
4,raw_aug_xgb_d5,raw,0.140625,2048,NaN,1.00,1.00,2.0,None
5,raw_lr_k2048_C0.1,raw,0.140625,2048,53274.0,1.00,1.00,2.0,[90]
6,gated_syn_down_mlp_a1,gated_syn_down,0.140625,1024,134554.0,0.25,0.25,2.0,114
7,gated_s3_aug_global_lgb4096,gated_syn_down_s3,0.250000,4096,NaN,0.25,0.25,3.0,300


## 6. 독립 재로딩·검증·문서 생성

In [10]:
import subprocess,tempfile,importlib.metadata
program="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);p=b['predict'](b,x)
np.save(sys.argv[3],p)
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 reloadp=Path(td)/'p.npy'
 result=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((DATA/'test.csv').resolve()),str(reloadp)],cwd=td,env=env,capture_output=True,text=True)
 if result.returncode:raise RuntimeError(result.stderr)
 assert np.allclose(np.load(reloadp),tp)
assert np.allclose(predict_bundle(loaded,test.iloc[:8]),tp[:8])
assert not set(groups[dev]).intersection(groups[hold])
verification=dict(status='passed',version=6,accepted_new_candidate=accepted6,primary_winner=winner6,primary_metrics=rank6.loc[winner6].to_dict(),baseline_primary=rank6.loc['v5_reference'].to_dict(),confirmation=conf6.to_dict(orient='index'),holdout=metrics,model_bytes=model_path.stat().st_size,full_independent_test_reload='passed',batch_invariance='passed',profile_split_overlap=0,public_score=None,python=sys.version,packages={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle']})
(OUT/'verification.json').write_text(json.dumps(verification,indent=2,ensure_ascii=False))
oldcsv=SUBMIT/'혜진_submission_v5.csv'
changed=int((pd.read_csv(oldcsv).SUBCLASS!=submission.SUBCLASS).sum()) if oldcsv.exists() else None
report=f"""# 혜진 v6 실험 보고서

팀원 문서를 참고해 v5 고정 대조군과 신규 설정 {len(NEW)}개를 같은 3-fold 그룹 CV에서 비교했습니다. 이전 59개를 재탐색하지 않았습니다. 원본 프로필을 그룹으로 묶고 기존 홀드아웃 1,241명을 개발 CV에서 제외했습니다.

## 검증 결과
채택 여부: **{accepted6}**. 선택 후보: `{winner6}`. 최종 구성: `{final_plan6}`.

| 지표 | v5 | 선택 후보 |
| --- | ---: | ---: |
| 개발 CV macro F1 | {rank6.loc['v5_reference','clean']:.6f} | {rank6.loc[winner6,'clean']:.6f} |
| 개발 CV 희귀 변이 추가 F1 | {rank6.loc['v5_reference','noise']:.6f} | {rank6.loc[winner6,'noise']:.6f} |
| 개발 CV 선택 점수 | {rank6.loc['v5_reference','score']:.6f} | {rank6.loc[winner6,'score']:.6f} |
| 다른 분할 clean F1 | {conf6.loc['v5_reference','clean']:.6f} | {conf6.loc['v6_frozen','clean']:.6f} |
| 다른 분할 선택 점수 | {conf6.loc['v5_reference','score']:.6f} | {conf6.loc['v6_frozen','score']:.6f} |

최종 모델 기존 홀드아웃 macro F1 **{metrics['macro_f1']:.6f}**, accuracy **{metrics['accuracy']:.6f}**, weighted F1 {metrics['weighted_f1']:.6f}입니다. 이 홀드아웃과 개발 데이터는 반복 관측했으므로 선택 편향이 있습니다. 팀원의 다른 검증 분할 0.4965와 직접 우열을 판단할 수 없습니다. **v6 Public 점수는 아직 모릅니다.** v4 Public은 사용자 제공 0.3169555443, v5 Public은 아직 제공되지 않았습니다.

## 적용 근거와 한계
팀원 문서의 직접 XGBoost/LightGBM 입력과 절대 변이 개수 보존은 v5에 이미 있었습니다. 새로 global 4,096개, 암종별 선별 2,048/4,096개, 학습 라벨에 따른 KL 특이도 가중치를 비교했습니다. 자세한 정의는 노트북 2절에 있습니다. 학습 데이터에서 변별력이 크다는 사실은 임상적 드라이버/병원성 증명이 아닙니다.

암종별 선별은 one-vs-rest 양성 방향 chi2 순위를 번갈아 채택하고, 신규 선별에서는 학습 열이 완전히 같은 경우만 중복 제거합니다. 전역 상관 임계값 0.98은 v5 구성에만 적용됩니다. 숫자 요약은 신규에서 강제로 보존합니다. specificity 후보는 변이 부분만 정규화하고 원시 변이 부담 등 숫자 요약은 별도로 보존합니다.

모든 사전·지원도·선별·특이도·스케일러는 해당 학습 fold로만 적합합니다. test 빈도나 정답은 사용하지 않습니다. test에 많은 변이를 일반인 유래라고 확정하지 않고, 위치 ±3을 무조건 같게 보지 않으며, class 비율 강제 맞추기를 하지 않습니다. 570개 예측이 모두 정답이라는 주장은 테스트 정답 없이는 확인 불가입니다.

## 실제 최종 변수
전체 목록: artifacts/v6/final_features.csv. 모델별 이름·가중치·입력 수는 model_sizes.json에 있습니다. 이전 v5 혼합을 포함하는 경우 기존 구성의 파싱·Pearson 선별·지원도 설정도 유지됩니다. 각 모델의 입력 수를 합산하면 {sum(s['inputs'] for s in sizes):,}개이고, 이는 고유 유전자 수가 아닙니다. 신규 모델을 추가할수록 계산량과 크기가 증가할 수 있습니다.

모델 {model_path.stat().st_size/1024**2:.2f} MiB, v5 대비 예측 변경 {changed}개입니다. 변경 개수는 정답 증가 수가 아닙니다. 피클은 프로젝트 코드 없는 임시 폴더에서 전체 test 예측을 재생성했고 확률·CSV·ID·배치 일치를 확인했습니다. 원본 데이터와 설치 환경은 필요합니다.

## 파일
cencer_v6.ipynb: 모든 실행 코드. requirements-v6.txt: 재현 환경. artifacts/v6/cancer_model_v6.pkl: 저장 모델. submission/혜진_submission_v6.csv: 제출 후보. 기존 CSV는 덮어쓰지 않습니다.
"""
(WORK/'analysis_report_v6.md').write_text(report)
meaning={'gene':'해당 유전자에서 관측된 변이의 가중 존재 여부','variant':'유전자와 정규화한 정확한 단백질 변이','position':'유전자와 단백질 위치; 다른 위치를 강제 일치시키지 않음','type':'유전자와 변이 유형','aa':'유전자와 아미노산 변화 종류; 서로 다른 위치도 같은 AA 변화 가능','supported':'학습에서 지지된 비동의 변이 존재; 임상적 병원성 아님','numeric':'변이 부담 로그·변이 유형 비율 등의 행별 요약','module':'명시한 경로 유전자 목록의 변이 가중합 log1p','missing':'결측 표시(기존 표현)'}
features['family']=features.feature.str.split(':').str[0];features['meaning']=features.family.map(meaning).fillna('노트북 정의 참조')
features.to_csv(OUT/'final_feature_dictionary.csv',index=False)
lines=['# v6 실제 최종 변수와 의미','','정확한 모델별 전체 변수는 artifacts/v6/final_feature_dictionary.csv에 있습니다.','', '| 계열 | 의미 |','| --- | --- |']+[f'| {k} | {v} |' for k,v in meaning.items()]
lines+=['','## 모델별 입력과 선정 근거','']
for s in sizes:lines.append(f"- `{s['model']}`: {s['inputs']:,}개, 앙상블 가중치 {s['weight']:.6f}. 해당 후보 설정과 CV 점수로 선택했습니다.")
lines+=['','선택된 모든 변수에 대해 개별적인 인과성·의학적 유효성을 확인한 것은 아닙니다. 수치적 선별 기준과 전체 목록을 공개합니다.','', '## 후보별 원본 설정','',json.dumps(NEW,indent=2,ensure_ascii=False)]
(WORK/'selected_variables_v6.md').write_text('\n'.join(lines));(WORK/'gene_dictionary_v6.md').write_text('\n'.join(lines[:len(meaning)+6]))
req='\n'.join(k+'=='+v for k,v in verification['packages'].items())+'\nmatplotlib=='+importlib.metadata.version('matplotlib')+'\nnbformat=='+importlib.metadata.version('nbformat')+'\nipython=='+importlib.metadata.version('ipython')+'\nthreadpoolctl=='+importlib.metadata.version('threadpoolctl')+'\n'
(WORK/'requirements-v6.txt').write_text(req)
print('Verification passed; report and artifacts saved.',flush=True)


Verification passed; report and artifacts saved.


## 최종 변수별 수치적 중요도
각 모델 안에서 중요도를 확인하는 진단입니다. 서로 다른 중요도 지표의 절대 크기를 비교하거나 인과효과로 해석하지 않습니다.

In [11]:
importance=[]
for m,w in zip(models,weights):
 if hasattr(m.est,'feature_importances_'):values=m.est.feature_importances_;kind_name='tree_importance'
 elif hasattr(m.est,'coef_'):values=np.max(np.abs(m.est.coef_),axis=0);kind_name='max_absolute_coefficient'
 else:values=np.linalg.norm(m.est.coefs_[0],axis=1);kind_name='first_layer_weight_norm'
 for feature,value in zip(m.names,values):importance.append(dict(model=m.name,view=m.view,feature=feature,importance=float(value),importance_kind=kind_name,ensemble_weight=float(w)))
importance=pd.DataFrame(importance);importance.to_csv(OUT/'final_feature_importance.csv',index=False)
top=importance.sort_values(['model','importance'],ascending=[True,False]).groupby('model').head(10)
top.to_csv(OUT/'top_features_by_model.csv',index=False)
path=WORK/'analysis_report_v6.md'
path.write_text(path.read_text()+'\n## EDA와 최종 변수 해석 파일\n\neda_by_class.csv에는 암종별 환자 수·변이 유전자 수 평균/중앙값/최대값, eda_mutated_gene_burden.csv에는 train/test 기술통계, eda_missing_by_gene.csv에는 유전자별 결측 수가 있습니다. test 통계는 설명에만 사용합니다.\n\nfinal_feature_importance.csv는 실제 최종 변수별 트리 중요도, 최대 절대 선형 계수 또는 신경망 첫 층 가중치 norm입니다. 모델 사이에서 수치 크기를 직접 비교할 수 없으며, 임상적 인과성이나 개별 환자의 기여도를 뜻하지 않습니다. 상위 10개씩은 top_features_by_model.csv에 있습니다.\n')
print('Detailed EDA, feature meanings and fitted feature importance saved.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
comparison=pd.read_csv(OUT/'model_comparison.csv',index_col=0)
show=comparison.head(7)
if 'v5_reference' not in show.index:show=pd.concat([show,comparison.loc[['v5_reference']]])
fig,ax=plt.subplots(figsize=(12,6));xx=np.arange(len(show));ax.barh(xx-.18,show.clean,height=.35,label='Clean macro F1');ax.barh(xx+.18,show.noise,height=.35,label='Rare-variant stress macro F1')
ax.set_yticks(xx,show.index);ax.invert_yaxis();ax.set_xlabel('Mean group 3-fold macro F1');ax.legend();fig.tight_layout();fig.savefig(OUT/'validation_comparison_v6.png',dpi=160);plt.close(fig)
path.write_text(path.read_text()+'\n![Development CV comparison](artifacts/v6/validation_comparison_v6.png)\n')

# Rebuild complete gene and final-variable Markdown dictionaries, including the new member.
fams={'gene','variant','position','type','aa','supported','missing'}
selected=importance.copy();selected['gene']=selected.feature.map(lambda f:f.split(':')[1] if ':' in f and f.split(':')[0] in fams else '')
counts=selected[selected.gene.ne('')].groupby('gene').size()
views=selected[selected.gene.ne('')].groupby('gene').view.agg(lambda z:', '.join(sorted(set(z))))
mut=(train[genes].notna()&train[genes].ne('WT'));te=(test[genes].notna()&test[genes].ne('WT'))
gdict=pd.DataFrame({'gene':genes,'train_mutated':mut.sum().values,'train_rate':mut.mean().values,'test_rate':te.mean().values})
gdict['selected_model_feature_uses']=gdict.gene.map(counts).fillna(0).astype(int);gdict['selected_views']=gdict.gene.map(views).fillna('-')
gdict.to_csv(OUT/'gene_dictionary_v6.csv',index=False)
glines=['# v6 전체 유전자 컬럼 사전','','관측 통계이며 유전자 기능/병원성 주석은 원본에 없어 추정하지 않습니다. test_rate는 기술통계로만 사용했습니다. selected_model_feature_uses는 모델 간 중복을 포함합니다. 0개라도 numeric/module 요약에 기여할 수 있습니다.','','| gene | train_mutated | train_rate | test_rate | selected_model_feature_uses | selected_views |','| --- | ---: | ---: | ---: | ---: | --- |']
for r in gdict.itertuples(index=False):glines.append(f'| {r.gene} | {r.train_mutated} | {r.train_rate:.5f} | {r.test_rate:.5f} | {r.selected_model_feature_uses} | {r.selected_views} |')
(WORK/'gene_dictionary_v6.md').write_text('\n'.join(glines)+'\n')
meaning={'gene':'유전자 변이 존재의 가중값','variant':'정규화한 정확한 단백질 변이','position':'유전자와 단백질 위치','type':'유전자와 변이 유형','aa':'유전자와 아미노산 변화 종류','supported':'학습에서 지지된 비동의 변이 존재','numeric':'변이 부담 로그 또는 변이 유형 비율 요약','module':'명시한 경로 유전자 변이 가중합 log1p','missing':'결측 여부'}
specific={'log_raw_genes':'log1p(변이 유전자 수)','log_effective_genes':'log1p(유전자별 최대 변이 가중치의 합)','log_nonsyn_genes':'log1p(유전자별 비동의 변이 최대 가중치의 합)','log_trunc_genes':'log1p(유전자별 stop/fs/del/ins 최대 가중치의 합)','log_multikind_genes':'log1p(변이 유형이 둘 이상인 유전자 수)','log_mutated_genes':'기존 표현: log1p(변이 유전자 수)','log_events':'기존 표현: log1p(공백으로 분리한 전체 변이 토큰 수; 중복 포함)','multi_event_excess':'기존 표현: log1p(max(변이 토큰 수-변이 유전자 수,0))','log_mean_position':'기존 표현: log1p(변이 위치 숫자 평균)'}
def feature_meaning(f):
 parts=f.split(':');family=parts[0];tail=parts[-1]
 if family=='numeric':
  if tail in specific:return specific[tail]
  if tail.startswith('gene_fraction_'):return '해당 유형의 유전자별 최대 가중치 합 / 변이 유전자 수; 유형 간 중복 가능'
  if tail.startswith('fraction_'):return '기존 표현: 해당 유형 토큰 수 / 전체 변이 토큰 수'
  if tail.startswith('log_'):return '기존 표현: log1p(해당 유형 변이 토큰 수)'
 return meaning.get(family,'표현 정의 참조')
slines=['# v6 최종 학습 변수 전체 목록','','모델별 실제 입력 목록입니다. 같은 변수도 view와 모델에 따라 의미/가중치가 다릅니다. 선정 근거는 고정 v5 대조군의 기존 선별, 또는 v6 학습 내 ANOVA/암종별 chi2/완전 동일 열 제거입니다. 숫자·경로 요약은 v6에서 항상 보존했습니다. 임상적 인과성은 확인하지 않았습니다. 중요도는 모델 안에서만 비교하며 서로 다른 지표를 비교할 수 없습니다.','','| model | view | feature | meaning | importance | metric |','| --- | --- | --- | --- | ---: | --- |']
for r in selected.itertuples(index=False):slines.append(f'| {r.model} | {r.view} | {r.feature} | {feature_meaning(r.feature)} | {r.importance:.7g} | {r.importance_kind} |')
(WORK/'selected_variables_v6.md').write_text('\n'.join(slines)+'\n')


Detailed EDA, feature meanings and fitted feature importance saved.
